In [1]:
import json
import pandas as pd


In [ ]:
volumes_path = "/home/ebr/projects/release-volume-sampler/generated/messina_001/volumes/recursive_propagations.json"
# Load the JSON file
with open(volumes_path, "r") as f:
    data = json.load(f)

In [4]:
data

[{'seed_triangle': 123,
  'seed_triangle_probability': 0.171000356265164,
  'volumes': [{'released': [123],
    'probability': 0.9412163938371734,
    'steps': [0],
    'area': 105192.69858619523},
   {'released': [123, 1909],
    'probability': 0.05861685479032843,
    'steps': [0, 1],
    'area': 208524.89587136992},
   {'released': [123, 1909, 6722],
    'probability': 0.00016675137249823959,
    'steps': [0, 1, 2],
    'area': 318952.5300366659}]},
 {'seed_triangle': 720,
  'seed_triangle_probability': 0.11423751188980827,
  'volumes': [{'released': [720],
    'probability': 1.0,
    'steps': [0],
    'area': 103271.09686582936},
   {'released': [720, 75],
    'probability': 0.0,
    'steps': [0, 1],
    'area': 210121.86912197497}]},
 {'seed_triangle': 1856,
  'seed_triangle_probability': 0.12173597215380896,
  'volumes': [{'released': [1856],
    'probability': 0.3004159118777695,
    'steps': [0],
    'area': 113349.69166008465},
   {'released': [1856, 1893],
    'probability': 

In [5]:
# Normalize the "volumes" data within each entry
df = pd.json_normalize(
    data,
    record_path=["volumes"],  # Extract and flatten the "volumes" list
    meta=["seed_triangle", "seed_triangle_probability"]  # Include these fields as metadata
)

# Keep 'released' as a list and skip expanding 'steps'
# Drop 'steps' if it's not needed
df = df.drop(columns=["steps"])

print(df)


                         released  probability           area seed_triangle  \
0                           [123]     0.941216  105192.698586           123   
1                     [123, 1909]     0.058617  208524.895871           123   
2               [123, 1909, 6722]     0.000167  318952.530037           123   
3                           [720]     1.000000  103271.096866           720   
4                       [720, 75]     0.000000  210121.869122           720   
..                            ...          ...            ...           ...   
596         [15242, 15244, 14220]     0.005920  133095.467645         15242   
597                       [15243]     0.700464   33685.745554         15243   
598                [15243, 15242]     0.277494   67380.518471         15243   
599         [15243, 15242, 15244]     0.020269  107318.063314         15243   
600  [15243, 15242, 15244, 14220]     0.001773  166781.213199         15243   

    seed_triangle_probability  
0                  

In [7]:
df.head()

,released,probability,area,seed_triangle,seed_triangle_probability
0,[123],0.941216,105192.698586,123,0.171
1,"[123, 1909]",0.058617,208524.895871,123,0.171
2,"[123, 1909, 6722]",0.000167,318952.530037,123,0.171
3,[720],1.000000,103271.096866,720,0.114238
4,"[720, 75]",0.000000,210121.869122,720,0.114238


In [8]:
import rasterio
import numpy as np

def write_volume_to_file(tri_mask_path, volume_path, triangle_indices):
    """
    Creates a binary mask for specified triangles and writes it to a new raster file.
    
    Parameters:
    - tri_mask_path (str): Path to the input raster file containing triangle indices.
    - volume_path (str): Path to the output volume raster file.
    - triangle_indices (list of int): List of triangle indices to include in the volume.
    """
    with rasterio.open(tri_mask_path) as src:
        tri_mask = src.read(1)  # Read the triangle mask
        profile = src.profile  # Copy metadata to use in output

    # Create binary volume mask: 1 if pixel belongs to specified triangles, else 0
    volume_mask = np.isin(tri_mask, triangle_indices).astype(np.uint8)

    # Update profile for single-band, unsigned 8-bit data
    profile.update(dtype=rasterio.uint8, count=1)

    with rasterio.open(volume_path, 'w', **profile) as dst:
        dst.write(volume_mask, 1)  # Write the volume mask to the output file